In [1]:
!pip install streamlit pypdf google-generativeai requests beautifulsoup4 pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.3 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
from pypdf import PdfReader
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
import re
import os
from datetime import datetime

st.set_page_config(page_title="Fact-Check Agent", page_icon="🔍", layout="wide")
st.title("🔍 Fact-Check Agent")
st.markdown("**Truth Layer for Marketing Content** – Upload PDF for Automated Fact-Checking")

# === GEMINI API KEY ===
if "GEMINI_API_KEY" not in st.session_state:
    st.session_state.GEMINI_API_KEY = ""

GEMINI_API_KEY = st.text_input("Enter your Gemini API Key", value=st.session_state.GEMINI_API_KEY, type="password")
if GEMINI_API_KEY:
    st.session_state.GEMINI_API_KEY = GEMINI_API_KEY
    genai.configure(api_key=GEMINI_API_KEY)

def extract_text_from_pdf(pdf_file):
    try:
        reader = PdfReader(pdf_file)
        text = "".join(page.extract_text() or "" for page in reader.pages)
        return text
    except:
        return "Error reading PDF"

def extract_claims(text):
    prompt = f"""Extract 8-15 key factual claims containing stats, percentages, dates, numbers from this text.
    Return ONLY a Python list like: ["claim 1", "claim 2"]"""

    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt + "\n\nText:\n" + text[:10000])
        claims_text = response.text
        claims = re.findall(r'"([^"]+)"', claims_text)
        if not claims:
            claims = [line.strip() for line in claims_text.split('\n') if len(line.strip()) > 20]
        return [c for c in claims if len(c) > 15][:15]
    except:
        return re.findall(r'.*?(?:\d+%|\d{4}|\$\d+|[0-9,.]+).*?[.!?]', text)[:12]

def verify_claim(claim):
    try:
        query = claim.replace(" ", "+") + "+2025+OR+2026"
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(f"https://www.google.com/search?q={query}", headers=headers, timeout=8)
        soup = BeautifulSoup(resp.text, 'html.parser')
        snippet = soup.find("div", class_="VwiC3b") or soup.find("span", class_="hgKElc")
        snippet_text = snippet.get_text()[:600] if snippet else "No strong evidence"

        model = genai.GenerativeModel('gemini-1.5-flash')
        judge_prompt = f"""Claim: {claim}

Evidence: {snippet_text}

Classify as **Verified**, **Inaccurate**, or **False**.
Give short reason and correct fact if needed."""

        result = model.generate_content(judge_prompt)
        return result.text
    except Exception as e:
        return f"Verification error: {str(e)[:150]}"

# Main UI
st.info("Get free Gemini API Key → https://aistudio.google.com/app/apikey")

uploaded_file = st.file_uploader("Upload PDF (try Trap_Document.pdf)", type="pdf")

if uploaded_file and GEMINI_API_KEY:
    with st.spinner("Analyzing PDF... This may take 20-40 seconds"):
        text = extract_text_from_pdf(uploaded_file)
        claims = extract_claims(text)

        st.success(f"✅ Extracted {len(claims)} claims")

        for i, claim in enumerate(claims):
            with st.expander(f"Claim {i+1}: {claim[:90]}...", expanded=(i < 2)):
                st.write("**Claim:**", claim)
                result = verify_claim(claim)
                st.markdown(result)

    st.download_button("Download Report", text, f"factcheck_{datetime.now().strftime('%Y%m%d')}.txt")

else:
    st.warning("Please enter Gemini API Key and upload a PDF")

st.caption("Built for CogCulture Product Management Assessment")

Writing app.py


In [3]:
from google.colab import files

print("Upload your PDF files now:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


Upload your PDF files now:


Saving Marketing_Report_Real.pdf to Marketing_Report_Real.pdf
Saving Trap_Document.pdf to Trap_Document.pdf
Uploaded file "Marketing_Report_Real.pdf" with length 2081 bytes
Uploaded file "Trap_Document.pdf" with length 1984 bytes


In [5]:
from pyngrok import ngrok
import subprocess
import time
from google.colab import userdata


!pkill -f streamlit
!pkill -f ngrok


subprocess.Popen(["streamlit", "run", "app.py", "--server.headless", "true", "--server.port", "8501"])

time.sleep(8)


NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')


if NGROK_AUTH_TOKEN == 'NGROK_AUTH_TOKEN':
    print("WARNING: The ngrok authtoken in Colab Secrets is still the placeholder. Please update it.")
    print("Instructions: Click '🔑' in the left sidebar -> find NGROK_AUTH_TOKEN -> update its 'Value' to your actual ngrok token.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken retrieved and set from Colab Secrets.")

# Create tunnel
public_url = ngrok.connect(8501)
print("🚀 Your Fact-Check App is Live!")
print(public_url)
print("\nClick the link above. If it shows a warning, click 'Visit Site'.")

ngrok authtoken retrieved and set from Colab Secrets.
🚀 Your Fact-Check App is Live!
NgrokTunnel: "https://fragility-plenty-perm.ngrok-free.dev" -> "http://localhost:8501"

Click the link above. If it shows a warning, click 'Visit Site'.
